# Resampling: Monte Carlo and the Bootstrap

This notebook accompanies [Simulations in Statistical Inference](../../notes/resampling_and_model_assessment/resampling.md). We (1) estimate a standard error by Monte Carlo from a known population, (2) bootstrap a single sample, (3) compare three bootstrap confidence intervals, and (4) bootstrap the slope of a regression by residual and pairs resampling.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

np.random.seed(42)

## Monte Carlo standard error of the sample mean

With a known $N(100,15^2)$ population we can draw many independent samples of size 30. The standard deviation of the resulting means estimates the standard error; theory gives $15/\sqrt{30}$.

In [2]:
mu, sigma, n, B = 100, 15, 30, 1000
means = np.array([np.random.normal(mu, sigma, n).mean() for _ in range(B)])
se_mc = means.std(ddof=1)
print(f"Monte Carlo SE = {se_mc:.4f}, theory = {sigma/np.sqrt(n):.4f}")
plt.hist(means, bins=30, color="skyblue", edgecolor="black")
plt.axvline(mu, color="green", linestyle=":")
plt.title("Monte Carlo distribution of the sample mean"); plt.show()

Monte Carlo SE = 2.7208, theory = 2.7386


/tmp/ipykernel_84914/2758577043.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title("Monte Carlo distribution of the sample mean"); plt.show()


## Bootstrapping one sample

In practice only one sample exists. Resampling it with replacement (the plug-in principle) approximates the sampling distribution. We use a skewed (lognormal) sample and bootstrap the **median**, a statistic with no simple standard-error formula.

In [3]:
sample = np.random.lognormal(mean=3, sigma=0.6, size=40)
theta_hat = np.median(sample)
boot = np.array([np.median(np.random.choice(sample, size=sample.size, replace=True))
                 for _ in range(2000)])
se_boot = boot.std(ddof=1)
print(f"median = {theta_hat:.3f}, bootstrap SE = {se_boot:.3f}")

median = 18.064, bootstrap SE = 1.997


## Three bootstrap confidence intervals

The normal, percentile and pivotal (basic) intervals use the same bootstrap replicates but different formulas. For a skewed statistic they can differ noticeably.

In [4]:
alpha = 0.05
lo, hi = np.percentile(boot, [100 * alpha / 2, 100 * (1 - alpha / 2)])
normal = (theta_hat - 1.96 * se_boot, theta_hat + 1.96 * se_boot)
percentile = (lo, hi)
pivotal = (2 * theta_hat - hi, 2 * theta_hat - lo)
for name, ci in [("normal", normal), ("percentile", percentile), ("pivotal", pivotal)]:
    print(f"{name:10s} [{ci[0]:.3f}, {ci[1]:.3f}]")
plt.hist(boot, bins=30, color="skyblue", edgecolor="black")
plt.axvline(theta_hat, color="red", linestyle="--")
plt.title("Bootstrap distribution of the median"); plt.show()

normal     [14.150, 21.978]
percentile [15.123, 21.792]
pivotal    [14.337, 21.005]


/tmp/ipykernel_84914/3490886813.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title("Bootstrap distribution of the median"); plt.show()


## Bootstrapping a regression slope

Residual resampling holds $X$ fixed and re-draws the errors; pairs resampling re-draws whole $(X_i,Y_i)$ observations. With homoscedastic errors they agree with each other and with the classical OLS standard error.

In [5]:
N = 100
X = np.random.uniform(0, 10, N)
Y = 3 + 2 * X + np.random.normal(0, 1, N)
Xc = sm.add_constant(X)
fit = sm.OLS(Y, Xc).fit()
a_hat, b_hat = fit.params
res = fit.resid

b_resid = np.empty(1000); b_pairs = np.empty(1000)
for i in range(1000):
    Ystar = a_hat + b_hat * X + np.random.choice(res, N, replace=True)
    b_resid[i] = sm.OLS(Ystar, Xc).fit().params[1]
    idx = np.random.randint(0, N, N)
    b_pairs[i] = sm.OLS(Y[idx], Xc[idx]).fit().params[1]

print(f"slope = {b_hat:.4f}")
print(f"OLS SE = {fit.bse[1]:.4f}, residual-bootstrap SE = {b_resid.std(ddof=1):.4f}, "
      f"pairs-bootstrap SE = {b_pairs.std(ddof=1):.4f}")
plt.hist(b_resid, bins=30, alpha=0.6, label="residual")
plt.hist(b_pairs, bins=30, alpha=0.6, label="pairs")
plt.axvline(b_hat, color="red", linestyle="--"); plt.legend()
plt.title("Bootstrap distribution of the slope"); plt.show()

slope = 2.0342
OLS SE = 0.0392, residual-bootstrap SE = 0.0378, pairs-bootstrap SE = 0.0356


/tmp/ipykernel_84914/1475084637.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title("Bootstrap distribution of the slope"); plt.show()


## Takeaways

- Monte Carlo needs a known population; the bootstrap replaces it by the empirical distribution.
- The bootstrap SE of the slope is close to the OLS SE when the model assumptions hold.
- Resampling single observations is only valid for independent data; dependent data need block or model-based schemes.